# PECE2
This notebook is now a thin entry point. The implementation lives in `src/` and post-processing matching lives in `scripts/match_names.py`.

In [7]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.helper import load_test_image
from src.utils.pipeline import process_exam

print('Imports ready')

Imports ready


In [1]:
img = load_test_image()
res = process_exam(img, img_name='sample', debug=False, policy='ocr')
print('Detected squares:', len(res['squares']))
print('Filled cells:', sum(1 for r in res['results'] if r.get('filled')))
print('Preview text:', ''.join(r.get('letter', '') for r in res['results'] if r.get('filled'))[:40])
print('Preprocessing result size:', res['img_deskewed'].shape[1], 'x', res['img_deskewed'].shape[0])

import cv2
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

fig, ax = plt.subplots(1, 1, figsize=(14, 16))
ax.imshow(cv2.cvtColor(res['vis'], cv2.COLOR_BGR2RGB))
ax.set_title('Ink-density / classification overlay')
ax.axis('off')
plt.tight_layout()
plt.show()

report_path = Path('all_extracted_results_with_knn.csv')
if report_path.exists():
    df = pd.read_csv(report_path)
    preview_cols = []
    if 'exam_name' in df.columns:
        preview_cols.append('exam_name')
    if 'extracted_text' in df.columns:
        preview_cols.append('extracted_text')
    if 'lev1_name' in df.columns:
        df = df.assign(levenshtein=f"{df['lev1_name'].astype(str)} ({df['lev1_dist'].astype(str)})")
        preview_cols.append('levenshtein')
    if 'ham1_name' in df.columns:
        df = df.assign(hamming=f"{df['ham1_name'].astype(str)} ({df['ham1_dist'].astype(str)})")
        preview_cols.append('hamming')
    if 'knn1_name' in df.columns:
        df = df.assign(knn=f"{df['knn1_name'].astype(str)} ({df['knn1_dist'].astype(str)})")
        preview_cols.append('knn')

    print('\nMatcher results preview (grouped by method):')
    display(df[preview_cols].head(5))
else:
    print('\nMatcher results preview unavailable. Run scripts/match_names.py first.')

NameError: name 'load_test_image' is not defined

## Notes
- Use `scripts/process_single.py` or `scripts/run_qualitative_test.py` for end-to-end runs.
- Use `scripts/match_names.py` for the saved-text matching stage.
- Keep this notebook light and delegate logic to the modules in `src/`.